## Ascend Qwen3.8-27B forward-pass latency and caching diagnostics

Mirrors `forward-gpt-test.ipynb` but targets the Huawei Ascend gateway serving
Qwen3.8-27B. Tests basic request latency, prompt-prefix caching, forward-pass
style requests (with `continue_final_message` + `enable_thinking=False`), and
raw HTTP diagnostics to inspect what the vLLM-style gateway reports.

In [1]:
import asyncio
import json
import os
import random
import time

import requests
from openai import AsyncOpenAI, OpenAI

from dotenv import load_dotenv

load_dotenv()

# Bypass the corporate proxy for the Huawei Ascend gateway.
# NOTE: no trailing .* on .huawei.com — httpx treats ".*" as requiring more chars
# after ".com", so ".huawei.com.*" does NOT match "llm-api.noah.huawei.com".
os.environ["no_proxy"] = ".huawei.com,10.0.0.0/8,localhost,127.0.0.1"
os.environ["NO_PROXY"] = ".huawei.com,10.0.0.0/8,localhost,127.0.0.1"

API_KEY = os.environ.get("OPENAI_API_KEY")
if not API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")

BASE_URL = os.environ.get("OPENAI_BASE_URL", "http://llm-api.noah.huawei.com/v1")
MODEL = "Qwen3.8-27B"

NUM_TRIALS = 5
RANDOM_SEED = 20260722

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
async_client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL)

# Same long body as the GPT notebook: crosses the usual 1024-token prompt-caching
# threshold. The condition-specific label occurs near the beginning, so unique-label
# requests cannot reuse the long cached prefix.
LONG_BODY = "\n".join(
    f"Record {i:04d}: alpha beta gamma delta epsilon zeta eta theta; "
    f"the measurement protocol and reporting format remain constant."
    for i in range(100)
)


def calculate_cost(prompt_tokens: int | None, completion_tokens: int | None) -> float | None:
    """Apply the same token-cost formula as API_LLM._accumulate_cost()."""
    if prompt_tokens is None or completion_tokens is None:
        return None
    # Ascend model pricing is zero (internal gateway), but keep the formula for parity.
    return 0.0


def make_prompt(first_token: str, prefix_label: str, trial: int | str) -> str:
    """Build equal-structure prompts whose first differing token is the label."""
    return (
        f"{first_token}mm,mmmmmm prefix variant {prefix_label}.\n"
        f"{LONG_BODY}\n\n"
    )


def send_request(first_token: str, condition: str, prefix_label: str, trial: int | str) -> dict:
    prompt = make_prompt(first_token, prefix_label, trial)
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1,
        temperature=1,
    )
    elapsed = time.perf_counter() - start

    usage = getattr(response, "usage", None)
    prompt_tokens = getattr(usage, "prompt_tokens", None) if usage else None
    completion_tokens = getattr(usage, "completion_tokens", None) if usage else None
    details = getattr(usage, "prompt_tokens_details", None) if usage else None
    cache_hit = getattr(details, "cached_tokens", 0) or 0 if details else 0
    request_cost = calculate_cost(prompt_tokens, completion_tokens)
    cost_text = f"${request_cost:.6f}" if request_cost is not None else "N/A"

    print(
        f"{condition:7s} trial {str(trial):>6s}: {elapsed:.3f}s | "
        f"prompt: {prompt_tokens} | cache hit: {cache_hit} | "
        f"output: {completion_tokens} | cost: {cost_text}"
    )
    return {
        "condition": condition,
        "trial": trial,
        "latency": elapsed,
        "prompt_tokens": prompt_tokens,
        "cache_hit": cache_hit,
        "completion_tokens": completion_tokens,
        "cost": request_cost,
    }


def print_total_cost(results: list[dict]) -> None:
    """Estimate total cost from API-reported usage."""
    complete_rows = [row for row in results if row["cost"] is not None]
    if not complete_rows:
        print("\nCost unavailable: the provider returned no token usage.")
        return

    prompt_tokens = sum(row["prompt_tokens"] for row in complete_rows if row["prompt_tokens"])
    output_tokens = sum(row["completion_tokens"] for row in complete_rows if row["completion_tokens"])
    total_cost = sum(row["cost"] for row in complete_rows)

    print("\n================ TOTAL EXPERIMENT COST ================")
    print(f"Requests counted: {len(complete_rows)}/{len(results)}")
    print(f"Input tokens: {prompt_tokens}")
    print(f"Output tokens: {output_tokens}")
    print(f"Adapter-calculated total: ${total_cost:.6f}")
    print("Ascend gateway pricing: $0.00 (internal)")
    print("======================================================")

### Prompt-prefix caching experiment

Shared requests reuse label A (same prefix → should hit cache). Unique requests
use different labels (early mismatch → cache miss). Compares whether the Ascend
vLLM gateway does prefix caching like OpenAI does.

In [2]:
jobs = []

REPEAT_CACHE = 15
NO_CACHE = 15

for trial in range(1, REPEAT_CACHE + 1):
    jobs.append(("7788", "shared", "A", 0))

for trial in range(1, NO_CACHE + 1):
    jobs.append(("7788", "unique", chr(ord("A") + trial), trial))

print("\n--- Running prompt-prefix caching trials ---")
sync_results = [send_request(*job) for job in jobs]
print_total_cost(sync_results)


--- Running prompt-prefix caching trials ---
shared  trial      0: 288.536s | prompt: 2765 | cache hit: 0 | output: 1 | cost: $0.000000
shared  trial      0: 967.430s | prompt: 2765 | cache hit: 0 | output: 1 | cost: $0.000000
shared  trial      0: 762.277s | prompt: 2765 | cache hit: 0 | output: 1 | cost: $0.000000
shared  trial      0: 313.805s | prompt: 2765 | cache hit: 0 | output: 1 | cost: $0.000000


APITimeoutError: Request timed out.

### Zero-content round-trip latency

Sends empty user content with `max_tokens=1` to measure the bare round-trip
latency of the Ascend gateway, independent of prompt length or generation.

In [ ]:
def measure_zero_content_round_trip() -> dict:
    """Send empty user content and return timing and API usage information."""
    started = time.perf_counter()
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": ""}],
            max_tokens=1,
            temperature=0,
        )
    except Exception as exc:
        elapsed = time.perf_counter() - started
        return {
            "elapsed_seconds": elapsed,
            "prompt_tokens": None,
            "completion_tokens": None,
            "cost": None,
            "error": str(exc),
        }

    elapsed = time.perf_counter() - started
    usage = getattr(response, "usage", None)
    prompt_tokens = getattr(usage, "prompt_tokens", None)
    completion_tokens = getattr(usage, "completion_tokens", None)
    return {
        "elapsed_seconds": elapsed,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "cost": calculate_cost(prompt_tokens, completion_tokens),
        "error": None,
    }

In [ ]:
NUM_ITERS = 100

zero_content_results = []
for iteration in range(1, NUM_ITERS + 1):
    result = measure_zero_content_round_trip()
    result["iteration"] = iteration
    zero_content_results.append(result)
    if result["error"] is None:
        print(f"Iteration {iteration:>3}: {result['elapsed_seconds']:.6f}s")
    else:
        print(
            f"Iteration {iteration:>3}: request failed after "
            f"{result['elapsed_seconds']:.6f}s — {result['error']}"
        )

reported_costs = [row["cost"] for row in zero_content_results if row["cost"] is not None]
if reported_costs:
    total_experiment_cost = sum(reported_costs)
    print(
        f"\nTotal for-loop experiment cost: ${total_experiment_cost:.8f} "
        f"({len(reported_costs)}/{NUM_ITERS} requests with usage data)"
    )
else:
    print("\nTotal for-loop experiment cost unavailable: no token usage was reported.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

successful_results = [row for row in zero_content_results if row["error"] is None]
iterations = [row["iteration"] for row in successful_results]
latencies = [row["elapsed_seconds"] for row in successful_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(iterations, latencies, s=20, color="tab:blue")
ax1.set(title="Ascend zero-content round-trip latency", xlabel="Iteration", ylabel="Round-trip time (s)")
ax1.grid(True, alpha=0.3)

ax2.hist(latencies, bins="auto", color="tab:blue", edgecolor="black", alpha=0.8)
ax2.set(title="Latency distribution", xlabel="Round-trip time (s)", ylabel="Frequency")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

latencies_arr = np.array(latencies)
print(f"Mean: {np.mean(latencies_arr):.4f}s | Std: {np.std(latencies_arr):.4f}s | Median: {np.median(latencies_arr):.4f}s")
print(f"Min: {np.min(latencies_arr):.4f}s | Max: {np.max(latencies_arr):.4f}s | p95: {np.percentile(latencies_arr, 95):.4f}s")

### Forward-pass style requests

Tests the `forward()` path used by confidence scoring: the prompt ends with an
assistant message and the gateway continues it (`continue_final_message=True`,
`add_generation_prompt=False`, `enable_thinking=False`). This is what
`API_LLM.forward()` sends for indirect/verbal confidence and step-bootstrap.

In [ ]:
def send_forward_pass_request(condition: str, trial: int | str) -> dict:
    """Send a forward-pass style request mirroring API_LLM.forward().

    The prompt ends with an assistant turn; the gateway continues it with
    thinking disabled, returning logprobs for the first generated token.
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer with True or False."},
        {"role": "user", "content": "The sky is blue. Is this statement true?"},
        {"role": "assistant", "content": "The answer is: "},
    ]

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=20,
        temperature=0.0,
        logprobs=True,
        top_logprobs=20,
        extra_body={
            "continue_final_message": True,
            "add_generation_prompt": False,
            "chat_template_kwargs": {"enable_thinking": False},
        },
    )
    elapsed = time.perf_counter() - start

    choice = response.choices[0]
    content = choice.message.content or ""
    reasoning = getattr(choice.message, "reasoning_content", None) or ""

    logprobs_content = choice.logprobs.content if choice.logprobs else []
    first_token_logprobs = logprobs_content[0] if logprobs_content else None
    top_logprobs = (
        {lp.token: lp.logprob for lp in first_token_logprobs.top_logprobs}
        if first_token_logprobs else {}
    )

    usage = getattr(response, "usage", None)
    prompt_tokens = getattr(usage, "prompt_tokens", None) if usage else None
    completion_tokens = getattr(usage, "completion_tokens", None) if usage else None

    print(
        f"{condition:7s} trial {str(trial):>6s}: {elapsed:.3f}s | "
        f"content: {content[:50]!r} | "
        f"reasoning: {len(reasoning)} chars | "
        f"first token: {first_token_logprobs.token if first_token_logprobs else 'N/A'} | "
        f"prompt: {prompt_tokens} | output: {completion_tokens}"
    )
    return {
        "condition": condition,
        "trial": trial,
        "latency": elapsed,
        "content": content,
        "reasoning_len": len(reasoning),
        "first_token": first_token_logprobs.token if first_token_logprobs else None,
        "top_logprobs": top_logprobs,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
    }


NUM_FORWARD_TRIALS = 10
print(f"--- Running {NUM_FORWARD_TRIALS} forward-pass trials ---")
forward_results = [send_forward_pass_request("fwd", i) for i in range(1, NUM_FORWARD_TRIALS + 1)]

In [ ]:
fwd_latencies = [r["latency"] for r in forward_results]

plt.figure(figsize=(9, 5))
plt.hist(fwd_latencies, bins="auto", color="tab:orange", edgecolor="black", alpha=0.8)
plt.title("Ascend forward-pass latency distribution")
plt.xlabel("Round-trip time (seconds)")
plt.ylabel("Frequency")
plt.grid(axis="y", alpha=0.3)
plt.show()

fwd_arr = np.array(fwd_latencies)
print(f"Mean: {np.mean(fwd_arr):.4f}s | Std: {np.std(fwd_arr):.4f}s | Median: {np.median(fwd_arr):.4f}s")

### Raw HTTP diagnostic

Sends a raw `requests.post` to inspect the full response body, headers, and any
latency checkpoint metadata the Ascend vLLM gateway returns.

In [ ]:
diagnostic_api_key = os.environ.get("OPENAI_API_KEY")
if not diagnostic_api_key:
    raise RuntimeError("Set OPENAI_API_KEY before running this cell.")

diagnostic_base_url = os.environ.get("OPENAI_BASE_URL", "http://llm-api.noah.huawei.com/v1").rstrip("/")
diagnostic_url = f"{diagnostic_base_url}/chat/completions"

diagnostic_headers = {
    "Authorization": f"Bearer {diagnostic_api_key}",
    "Content-Type": "application/json",
}
diagnostic_payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Reply with exactly: OK"}],
    "max_tokens": 1,
    "temperature": 0,
}

started = time.perf_counter()
diagnostic_response = requests.post(
    diagnostic_url,
    headers=diagnostic_headers,
    json=diagnostic_payload,
    timeout=120,
)
elapsed = time.perf_counter() - started

print(f"HTTP status: {diagnostic_response.status_code}")
print(f"Client elapsed: {elapsed:.3f}s")
print(f"Server elapsed: {diagnostic_response.elapsed.total_seconds():.3f}s")
print("\nResponse headers:")
for name, value in sorted(diagnostic_response.headers.items()):
    print(f"  {name}: {value}")

print("\nResponse body:")
content = json.loads(diagnostic_response.content.decode("utf-8"))
print(json.dumps(content, indent=2))

In [ ]:
if "usage" in content:
    print("=== Usage keys ===")
    print(json.dumps(content["usage"], indent=2))
else:
    print("No usage field in response")

if "latency_checkpoint" in content.get("usage", {}):
    print("\n=== Latency checkpoint ===")
    print(json.dumps(content["usage"]["latency_checkpoint"], indent=2))
else:
    print("\nNo latency_checkpoint in usage (Ascend gateway may not report this)")

In [ ]:
def send_post_response(first_token: str, condition: str, prefix_label: str, trial: int | str) -> dict:
    prompt = make_prompt(first_token, prefix_label, trial)

    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1,
        "temperature": 1,
    }

    started = time.perf_counter()
    resp = requests.post(
        diagnostic_url,
        headers=diagnostic_headers,
        json=payload,
        timeout=120,
    )
    elapsed = time.perf_counter() - started

    body = json.loads(resp.content.decode("utf-8"))
    usage = body.get("usage", {})

    prompt_tokens = usage.get("prompt_tokens")
    completion_tokens = usage.get("completion_tokens")
    cache_hit = usage.get("prompt_tokens_details", {}).get("cached_tokens", 0)
    server_elapsed = resp.elapsed.total_seconds()

    latency_checkpoint = usage.get("latency_checkpoint", {})
    engine_ttlt_ms = latency_checkpoint.get("engine_ttlt_ms", "N/A")

    print(
        f"{condition:7s} trial {str(trial):>6s}: client {elapsed:.3f}s | server {server_elapsed:.3f}s | "
        f"engine {engine_ttlt_ms}ms | "
        f"prompt: {prompt_tokens} | cache hit: {cache_hit} | output: {completion_tokens}"
    )
    return {
        "condition": condition,
        "trial": trial,
        "client_elapsed": elapsed,
        "server_elapsed": server_elapsed,
        "engine_ttlt_ms": engine_ttlt_ms,
        "prompt_tokens": prompt_tokens,
        "cached_tokens": cache_hit,
        "completion_tokens": completion_tokens,
    }


jobs = []
REPEAT_CACHE = 15
NO_CACHE = 15

for trial in range(1, REPEAT_CACHE + 1):
    jobs.append(("h723", "shared", "A", 0))
for trial in range(1, NO_CACHE + 1):
    jobs.append(("h723", "unique", chr(ord("A") + trial), trial))

print("\n--- Running raw HTTP caching trials ---")
post_sync_results = [send_post_response(*job) for job in jobs]

In [ ]:
shared_latencies = [r["client_elapsed"] for r in post_sync_results if r["condition"] == "shared"]
unique_latencies = [r["client_elapsed"] for r in post_sync_results if r["condition"] == "unique"]
shared_cache = [r["cached_tokens"] for r in post_sync_results if r["condition"] == "shared"]
unique_cache = [r["cached_tokens"] for r in post_sync_results if r["condition"] == "unique"]

print("=== Summary ===")
print(f"Shared:  mean {np.mean(shared_latencies):.3f}s | cache hits: {shared_cache}")
print(f"Unique:  mean {np.mean(unique_latencies):.3f}s | cache hits: {unique_cache}")
print()
print("If the Ascend gateway supports prefix caching, shared should be faster")
print("with nonzero cache hits, while unique should show cache misses.")